**Сбор данных**

a. Процентные ставки на разные сроки (от 0 до 30 лет) за период с 1 января 2021 г. по 1 января 2026 г.

b. Описания 5 государственных облигаций РФ (расписания выплат). Критерии —
государственные облигации с полностью известными размерами выплат (не
привязанные к показателям), без оферт, со сроком погашения после 1 января
2026 г.

c. Рыночные котировки этих облигаций за период с 1 января 2021 г. по 1 января
2026 г.

d. Котировки 10 российских акций за тот же период.

e. Значения индекса МосБиржи, индекса РТС, цены на нефть Brent и курса доллара и
евро за тот же период.

f. Котировки фьючерса и опционов на фьючерс на один выбранный актив из
предыдущего пункта — выбрать один торговый день за 2025 год. Срок погашения и
фьючерса, и опционов взять ближайшие к выбранному дню, но не ближе, чем 1
месяц.
Опционы и Put, и Call — только для бонусного задания.

In [1]:
#  импорты, конфигурация, доступ к API MOEX
import requests
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# даты для проекта
START, END = "2021-01-01", "2026-01-01"

# базовый API Московской биржи
ISS = "https://iss.moex.com/iss"

#  автоповторы
def _make_session():
    s = requests.Session()
    retry = Retry(total=5, backoff_factor=0.6,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=["GET"])
    s.mount("https://", HTTPAdapter(max_retries=retry, pool_maxsize=16))
    s.headers.update({"User-Agent": "risk-mgmt-coursework/1.0"})
    return s

SESSION = _make_session()

def iss_get(path, **params):
    params.setdefault("iss.meta", "off")
    r = SESSION.get(f"{ISS}{path}.json", params=params, timeout=60)
    r.raise_for_status()
    return r.json()

# Постраничная выгрузка
def iss_paginate(path, block="history", **params):
    params.setdefault("iss.only", block)   # просим только нужный блок
    params.setdefault("limit", 100)
    frames, start = [], 0
    while True:
        params["start"] = start
        d = iss_get(path, **params)[block]
        rows = d["data"]
        if not rows:
            break
        frames.append(pd.DataFrame(rows, columns=d["columns"]))
        start += len(rows)
        if len(rows) < params["limit"]:    # последняя страница
            break
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

_test = iss_get("/engines")["engines"]["data"]
print("API доступен. Торговых систем MOEX:", len(_test))

API доступен. Торговых систем MOEX: 11


In [2]:
def iss_history(engine, market, secid, board=None, columns=None,
                start=START, end=END):
    path = f"/history/engines/{engine}/markets/{market}"
    if board:
        path += f"/boards/{board}"
    path += f"/securities/{secid}"
    p = {"from": start, "till": end}
    if columns:
        p["history.columns"] = ",".join(columns)
    df = iss_paginate(path, "history", **p)
    if not df.empty and "TRADEDATE" in df.columns:
        df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
        # числовые колонки — к числам (из JSON иногда приходят строками)
        for c in df.columns:
            if c != "TRADEDATE":
                df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# Шаг 1. Дневная история КБД с сайта ЦБ РФ за 2021–2026

In [3]:

import io, re
from concurrent.futures import ThreadPoolExecutor, as_completed

CBR_ZCYC = "https://www.cbr.ru/hd_base/zcyc_params/zcyc/"

def _cbr_curve_on(d):
    """Кривая на дату d. Возвращает (реальная_дата_ЦБ, Series ставок по срокам)."""
    r = SESSION.get(CBR_ZCYC, params={"DateTo": d.strftime("%d.%m.%Y")}, timeout=30)
    r.raise_for_status()
    # дата, которую реально вернул ЦБ
    head = r.text.split("Срок до погашения")[0][-400:]
    dates = re.findall(r"\d{2}\.\d{2}\.\d{4}", head)
    real_date = pd.to_datetime(dates[-1], dayfirst=True) if dates else d
    tbl = pd.read_html(io.StringIO(r.text))[0]
    tenors = [float(str(c).replace(",", ".")) for c in tbl.columns[1:]]
    yields = pd.to_numeric(
        tbl.iloc[0, 1:].astype(str).str.replace(",", ".", regex=False),
        errors="coerce").to_numpy()
    return real_date, pd.Series(yields, index=[f"ZCYC_{t}y" for t in tenors])

def fetch_zcyc_cbr(start=START, end=END, workers=8):
    days = pd.bdate_range(start, end)                  # рабочие дни (Пн–Пт)
    out, done = {}, 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = [ex.submit(_cbr_curve_on, d) for d in days]
        for f in as_completed(futs):
            try:
                rd, s = f.result()
                out[rd] = s
            except Exception:
                pass                                   # пустые/нерабочие дни пропускаем
            done += 1
            if done % 200 == 0:
                print(f"  обработано: {done}/{len(days)}")
    curves = pd.DataFrame(out).T.sort_index()
    curves.index.name = "TRADEDATE"
    return curves

zcyc = fetch_zcyc_cbr()
zcyc.to_csv("zcyc.csv")
print("\nКБД (ЦБ):", zcyc.shape, "| даты:", zcyc.index.min(), "→", zcyc.index.max())
zcyc.tail(3)

  обработано: 200/1305
  обработано: 400/1305
  обработано: 600/1305
  обработано: 800/1305
  обработано: 1000/1305
  обработано: 1200/1305

КБД (ЦБ): (1305, 12) | даты: 2021-01-01 00:00:00 → 2026-01-01 00:00:00


,ZCYC_0.25y,ZCYC_0.5y,ZCYC_0.75y,ZCYC_1.0y,ZCYC_2.0y,ZCYC_3.0y,ZCYC_5.0y,ZCYC_7.0y,ZCYC_10.0y,ZCYC_15.0y,ZCYC_20.0y,ZCYC_30.0y
TRADEDATE,,,,,,,,,,,,
2025-12-30,12.1,12.5,12.84,13.14,13.92,14.3,14.58,14.59,14.44,14.11,13.91,13.79
2025-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Шаг 2. Отбор 5 ОФЗ под критерии задания

In [4]:
# ОФЗ-ПД (SU26*) фикс. купон, без оферт, погашение пулей

def list_ofz(board="TQOB"):
    # TQOB — основной режим торгов гособлигациями
    cols = "SECID,SHORTNAME,MATDATE,FACEVALUE,COUPONPERCENT,COUPONVALUE,OFFERDATE,BUYBACKDATE"
    d = iss_get(f"/engines/stock/markets/bonds/boards/{board}/securities",
                **{"iss.only": "securities", "securities.columns": cols})["securities"]
    df = pd.DataFrame(d["data"], columns=d["columns"])
    df["MATDATE"] = pd.to_datetime(df["MATDATE"], errors="coerce")
    return df

bonds = list_ofz()
print("Инструментов на доске TQOB:", len(bonds))

# ОФЗ-ПД с погашением после 01.01.2026
flt = (bonds[bonds["SECID"].str.startswith("SU26")
             & (bonds["MATDATE"] > pd.Timestamp("2026-01-01"))]
       .sort_values("MATDATE").reset_index(drop=True))
print("\nПодходящих выпусков ОФЗ-ПД:", len(flt))
print(flt[["SECID","SHORTNAME","MATDATE","COUPONPERCENT"]].to_string())

# --- выбираем 5, равномерно по срочности
# Разная срочность = разная дюрация = разный процентный риск
idx = np.linspace(0, len(flt) - 1, 5).round().astype(int)
OFZ = flt.loc[idx, "SECID"].tolist()
print("\nВыбраны 5 ОФЗ:", OFZ)
print(flt.loc[idx, ["SECID","SHORTNAME","MATDATE","COUPONPERCENT"]].to_string(index=False))



Инструментов на доске TQOB: 63

Подходящих выпусков ОФЗ-ПД: 33
           SECID  SHORTNAME    MATDATE  COUPONPERCENT
0   SU26219RMFS4  ОФЗ 26219 2026-09-16           7.75
1   SU26226RMFS9  ОФЗ 26226 2026-10-07           7.95
2   SU26207RMFS9  ОФЗ 26207 2027-02-03           8.15
3   SU26232RMFS7  ОФЗ 26232 2027-10-06           6.00
4   SU26212RMFS9  ОФЗ 26212 2028-01-19           7.05
5   SU26236RMFS8  ОФЗ 26236 2028-05-17           5.70
6   SU26237RMFS6  ОФЗ 26237 2029-03-14           6.70
7   SU26224RMFS4  ОФЗ 26224 2029-05-23           6.90
8   SU26242RMFS6  ОФЗ 26242 2029-08-29           9.00
9   SU26228RMFS5  ОФЗ 26228 2030-04-10           7.65
10  SU26251RMFS7  ОФЗ 26251 2030-08-28           9.50
11  SU26235RMFS0  ОФЗ 26235 2031-03-12           5.90
12  SU26239RMFS2  ОФЗ 26239 2031-07-23           6.90
13  SU26218RMFS6  ОФЗ 26218 2031-09-17           8.50
14  SU26249RMFS1  ОФЗ 26249 2032-06-16          11.00
15  SU26241RMFS8  ОФЗ 26241 2032-11-17           9.50
16  SU26221RMFS0  О

In [5]:
# Расписание выплат по каждой ОФЗ (купоны + погашение)

OFZ = ["SU26219RMFS4", "SU26212RMFS9", "SU26221RMFS0", "SU26218RMFS6", "SU26230RMFS1"]

def bond_schedule(secid):
    """Возвращает dict с DataFrame'ами: coupons, amortizations, offers."""
    res = {}
    for block in ["coupons", "amortizations", "offers"]:
        df = iss_paginate(f"/securities/{secid}/bondization", block,
                          **{"limit": 100})
        res[block] = df
    return res

schedules = {s: bond_schedule(s) for s in OFZ}

# сводка по каждой бумаге: сколько купонов, есть ли оферты, диапазон дат
print(f"{'SECID':<14}{'купонов':>8}{'погашений':>11}{'оферт':>7}   период выплат")
for s in OFZ:
    c = schedules[s]["coupons"]
    a = schedules[s]["amortizations"]
    o = schedules[s]["offers"]
    dates = pd.to_datetime(c["coupondate"], errors="coerce")
    print(f"{s:<14}{len(c):>8}{len(a):>11}{len(o):>7}   "
          f"{dates.min():%Y-%m-%d} → {dates.max():%Y-%m-%d}")

SECID          купонов  погашений  оферт   период выплат
SU26219RMFS4        21          1      0   2016-09-28 → 2026-09-16
SU26212RMFS9        30          1      0   2013-08-07 → 2028-01-19
SU26221RMFS0        32          1      0   2017-10-11 → 2033-03-23
SU26218RMFS6        32          1      0   2016-04-06 → 2031-09-17
SU26230RMFS1        40          1      0   2019-10-09 → 2039-03-16


In [6]:
# Сохраняем расписания выплат всех 5 ОФЗ в один файл
rows = []
for s in OFZ:
    c = schedules[s]["coupons"].copy()
    c["secid"] = s
    c["type"] = "coupon"
    c = c.rename(columns={"coupondate": "date"})
    am = schedules[s]["amortizations"].copy()
    am["secid"] = s
    am["type"] = "amortization"
    am = am.rename(columns={"amortdate": "date"})
    rows.append(c[["secid","date","value","type"]])
    rows.append(am[["secid","date","value","type"]])

cashflows = pd.concat(rows, ignore_index=True)
cashflows["date"] = pd.to_datetime(cashflows["date"])
cashflows = cashflows.sort_values(["secid","date"]).reset_index(drop=True)
cashflows.to_csv("ofz_cashflows.csv", index=False)
print("Сохранено платежей:", len(cashflows), "→ ofz_cashflows.csv")
print(cashflows.groupby("secid").size().rename("платежей"))

Сохранено платежей: 160 → ofz_cashflows.csv
secid
SU26212RMFS9    31
SU26218RMFS6    33
SU26219RMFS4    22
SU26221RMFS0    33
SU26230RMFS1    41
Name: платежей, dtype: int64


# Шаг 3.  Рыночные котировки 5 ОФЗ за 2021–2026

In [7]:
def ofz_quotes(secid):
    df = iss_history("stock", "bonds", secid, board="TQOB",
                     columns=["TRADEDATE","CLOSE","YIELDCLOSE","DURATION","ACCINT"])
    return secid, df.set_index("TRADEDATE").sort_index()

ofz_prices = {}
with ThreadPoolExecutor(max_workers=5) as ex:
    for f in as_completed([ex.submit(ofz_quotes, s) for s in OFZ]):
        secid, df = f.result()
        ofz_prices[secid] = df

# сколько дней истории и за какой период по каждой бумаге
print(f"{'SECID':<14}{'дней':>7}   период котировок        посл. цена / доходность")
for s in OFZ:
    df = ofz_prices[s]
    last = df.dropna(subset=["CLOSE"]).iloc[-1]
    print(f"{s:<14}{len(df):>7}   {df.index.min():%Y-%m-%d} → {df.index.max():%Y-%m-%d}   "
          f"{last['CLOSE']:.2f}% / {last['YIELDCLOSE']:.2f}%")

SECID            дней   период котировок        посл. цена / доходность
SU26219RMFS4     1271   2021-01-04 → 2025-12-30   97.00% / 12.72%
SU26212RMFS9     1271   2021-01-04 → 2025-12-30   88.52% / 14.15%
SU26221RMFS0     1271   2021-01-04 → 2025-12-30   72.03% / 14.46%
SU26218RMFS6     1271   2021-01-04 → 2025-12-30   79.30% / 14.34%
SU26230RMFS1     1271   2021-01-04 → 2025-12-30   63.83% / 14.16%


In [8]:
# Сборка и сохранение котировок ОФЗ

ofz_close = pd.concat(
    {s: ofz_prices[s]["CLOSE"] for s in OFZ}, axis=1
)
ofz_close.columns = [f"OFZ_{s[2:7]}" for s in OFZ]
ofz_close.index.name = "TRADEDATE"
ofz_close.to_csv("ofz_prices.csv")

long = []
for s in OFZ:
    d = ofz_prices[s].copy()
    d["secid"] = s
    long.append(d.reset_index())
ofz_full = pd.concat(long, ignore_index=True)
ofz_full.to_csv("ofz_quotes_full.csv", index=False)

print("Цены закрытия:", ofz_close.shape, "→ ofz_prices.csv")
print("Полные котировки:", ofz_full.shape, "→ ofz_quotes_full.csv")
print("\nПропуски в ценах по бумагам:")
print(ofz_close.isna().sum())
print("\nХвост таблицы цен:")
print(ofz_close.tail(3).round(2))

Цены закрытия: (1271, 5) → ofz_prices.csv
Полные котировки: (6355, 6) → ofz_quotes_full.csv

Пропуски в ценах по бумагам:
OFZ_26219    15
OFZ_26212    15
OFZ_26221    15
OFZ_26218    15
OFZ_26230    15
dtype: int64

Хвост таблицы цен:
            OFZ_26219  OFZ_26212  OFZ_26221  OFZ_26218  OFZ_26230
TRADEDATE                                                        
2025-12-26      96.37      89.17      71.35      78.76      63.10
2025-12-29      96.50      89.15      71.64      79.53      63.37
2025-12-30      97.00      88.52      72.03      79.30      63.83


# Шаг 4 10 российских акций (обыкновенные, ликвидные) за 2021–2026

In [9]:
STOCKS = ["SBER", "GAZP", "LKOH", "GMKN", "ROSN", "NVTK", "TATN", "MGNT", "MTSS", "CHMF"]

def stock_quotes(secid):
    df = iss_history("stock", "shares", secid, board="TQBR",
                     columns=["TRADEDATE","CLOSE","VOLUME"])
    return secid, df.set_index("TRADEDATE").sort_index()

stock_data = {}
with ThreadPoolExecutor(max_workers=10) as ex:
    for f in as_completed([ex.submit(stock_quotes, s) for s in STOCKS]):
        secid, df = f.result()
        stock_data[secid] = df

# сводка по каждой бумаге
print(f"{'TICKER':<8}{'дней':>7}   период котировок        посл. цена")
for s in STOCKS:
    df = stock_data[s]
    last = df.dropna(subset=["CLOSE"]).iloc[-1]
    print(f"{s:<8}{len(df):>7}   {df.index.min():%Y-%m-%d} → {df.index.max():%Y-%m-%d}   "
          f"{last['CLOSE']:.2f}")

TICKER     дней   период котировок        посл. цена
SBER       1271   2021-01-04 → 2025-12-30   299.90
GAZP       1271   2021-01-04 → 2025-12-30   125.36
LKOH       1271   2021-01-04 → 2025-12-30   5908.50
GMKN       1271   2021-01-04 → 2025-12-30   149.46
ROSN       1271   2021-01-04 → 2025-12-30   409.00
NVTK       1271   2021-01-04 → 2025-12-30   1188.00
TATN       1271   2021-01-04 → 2025-12-30   580.70
MGNT       1271   2021-01-04 → 2025-12-30   3012.00
MTSS       1271   2021-01-04 → 2025-12-30   213.90
CHMF       1271   2021-01-04 → 2025-12-30   962.20


In [10]:
# Сборка и сохранение котировок акций

stocks_close = pd.concat({s: stock_data[s]["CLOSE"] for s in STOCKS}, axis=1)
stocks_close.index.name = "TRADEDATE"
stocks_close.to_csv("stocks_prices.csv")

print("Цены акций:", stocks_close.shape, "→ stocks_prices.csv")
print("\nПропуски по бумагам:")
print(stocks_close.isna().sum())

# Проверим, совпадают ли пропуски с заморозкой 2022 (как у ОФЗ)
gaps = stocks_close[stocks_close.isna().all(axis=1)].index
print("\nДней с пропуском у всех акций:", len(gaps))
print(gaps.tolist() if len(gaps) else "нет общих пропусков")

print("\nХвост таблицы цен:")
print(stocks_close.tail(3).round(2))

Цены акций: (1271, 10) → stocks_prices.csv

Пропуски по бумагам:
SBER    18
GAZP    18
LKOH    18
GMKN    22
ROSN    18
NVTK    18
TATN    18
MGNT    18
MTSS    18
CHMF    18
dtype: int64

Дней с пропуском у всех акций: 18
[Timestamp('2022-01-07 00:00:00'), Timestamp('2022-02-23 00:00:00'), Timestamp('2022-02-28 00:00:00'), Timestamp('2022-03-01 00:00:00'), Timestamp('2022-03-02 00:00:00'), Timestamp('2022-03-03 00:00:00'), Timestamp('2022-03-04 00:00:00'), Timestamp('2022-03-09 00:00:00'), Timestamp('2022-03-10 00:00:00'), Timestamp('2022-03-11 00:00:00'), Timestamp('2022-03-14 00:00:00'), Timestamp('2022-03-15 00:00:00'), Timestamp('2022-03-16 00:00:00'), Timestamp('2022-03-17 00:00:00'), Timestamp('2022-03-18 00:00:00'), Timestamp('2022-03-21 00:00:00'), Timestamp('2022-03-22 00:00:00'), Timestamp('2022-03-23 00:00:00')]

Хвост таблицы цен:
              SBER    GAZP    LKOH    GMKN    ROSN    NVTK   TATN    MGNT  \
TRADEDATE                                                          

# Шаг 5. Индексы МосБиржи (IMOEX) и РТС (RTSI) за 2021–2026

In [11]:
INDICES = ["IMOEX", "RTSI"]

def index_quotes(secid):
    df = iss_history("stock", "index", secid, columns=["TRADEDATE","CLOSE"])
    return secid, df.set_index("TRADEDATE")["CLOSE"].sort_index()

index_data = {}
with ThreadPoolExecutor(max_workers=2) as ex:
    for f in as_completed([ex.submit(index_quotes, s) for s in INDICES]):
        secid, ser = f.result()
        index_data[secid] = ser

indices_df = pd.concat(index_data, axis=1)
indices_df.index.name = "TRADEDATE"

print(f"{'INDEX':<8}{'дней':>7}   период                  посл. значение")
for s in INDICES:
    ser = index_data[s].dropna()
    print(f"{s:<8}{len(index_data[s]):>7}   "
          f"{ser.index.min():%Y-%m-%d} → {ser.index.max():%Y-%m-%d}   {ser.iloc[-1]:.2f}")

print("\nХвост:")
print(indices_df.tail(3).round(2))

INDEX      дней   период                  посл. значение
IMOEX      1254   2021-01-04 → 2025-12-30   2766.62
RTSI       1253   2021-01-04 → 2025-12-30   1114.13

Хвост:
               RTSI    IMOEX
TRADEDATE                   
2025-12-26  1117.04  2754.89
2025-12-29  1115.35  2742.03
2025-12-30  1114.13  2766.62


In [12]:
# Курсы USD/RUB и EUR/RUB — официальный курс ЦБ РФ

CBR_DYN = "https://www.cbr.ru/scripts/XML_dynamic.asp"
FX_CODES = {"USD": "R01235", "EUR": "R01239"}

def cbr_fx(name, code, start=START, end=END):
    r = SESSION.get(CBR_DYN, params={
        "date_req1": pd.Timestamp(start).strftime("%d/%m/%Y"),
        "date_req2": pd.Timestamp(end).strftime("%d/%m/%Y"),
        "VAL_NM_RQ": code,
    }, timeout=30)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    recs = []
    for rec in root.findall("Record"):
        date = pd.to_datetime(rec.get("Date"), dayfirst=True)
        nominal = float(rec.find("Nominal").text.replace(",", "."))
        value = float(rec.find("Value").text.replace(",", "."))
        recs.append((date, value / nominal))      # курс за 1 единицу
    return (pd.DataFrame(recs, columns=["TRADEDATE", name])
              .set_index("TRADEDATE").sort_index())

fx_df = pd.concat([cbr_fx(name, code) for name, code in FX_CODES.items()], axis=1)
fx_df.index.name = "TRADEDATE"
fx_df.to_csv("fx_prices.csv")

print(f"{'VALUTA':<6}{'дней':>7}   период                  посл. курс")
for name in FX_CODES:
    ser = fx_df[name].dropna()
    print(f"{name:<6}{len(ser):>7}   "
          f"{ser.index.min():%Y-%m-%d} → {ser.index.max():%Y-%m-%d}   {ser.iloc[-1]:.4f} ₽")

print("\nХвост:")
print(fx_df.tail(3).round(4))

VALUTA   дней   период                  посл. курс
USD      1237   2021-01-01 → 2025-12-31   78.2267 ₽
EUR      1237   2021-01-01 → 2025-12-31   92.0938 ₽

Хвост:
                USD      EUR
TRADEDATE                   
2025-12-27  77.6923  91.2066
2025-12-30  77.4466  91.4775
2025-12-31  78.2267  92.0938


In [13]:
# Brent

def discover_br_contracts(start=START, end=END):
    secids = set()
    for d in pd.date_range(start, end, freq="MS"):       # 1-е число каждого месяца
        try:
            js = iss_get("/history/engines/futures/markets/forts/securities",
                         date=d.strftime("%Y-%m-%d"))
            df = pd.DataFrame(js["history"]["data"], columns=js["history"]["columns"])
            secids |= set(df.loc[df["ASSETCODE"] == "BR", "SECID"])
        except Exception:
            pass
    return sorted(secids)

br_contracts = discover_br_contracts()
print("Найдено контрактов BR:", len(br_contracts))
print(br_contracts)

Найдено контрактов BR: 66
['BRF2', 'BRF3', 'BRF4', 'BRF5', 'BRF6', 'BRG1', 'BRG2', 'BRG3', 'BRG4', 'BRG5', 'BRG6', 'BRH1', 'BRH2', 'BRH3', 'BRH4', 'BRH5', 'BRH6', 'BRJ1', 'BRJ2', 'BRJ3', 'BRJ4', 'BRJ5', 'BRJ6', 'BRK1', 'BRK2', 'BRK3', 'BRK4', 'BRK5', 'BRK6', 'BRM1', 'BRM2', 'BRM3', 'BRM4', 'BRM5', 'BRM6', 'BRN1', 'BRN2', 'BRN3', 'BRN4', 'BRN5', 'BRN6', 'BRQ1', 'BRQ2', 'BRQ3', 'BRQ4', 'BRQ5', 'BRU1', 'BRU2', 'BRU3', 'BRU4', 'BRU5', 'BRV1', 'BRV2', 'BRV3', 'BRV4', 'BRV5', 'BRX1', 'BRX2', 'BRX3', 'BRX4', 'BRX5', 'BRZ1', 'BRZ2', 'BRZ3', 'BRZ4', 'BRZ5']


In [14]:
def br_contract_history(secid):
    df = iss_history("futures", "forts", secid,
                     columns=["TRADEDATE","CLOSE","SETTLEPRICE","VOLUME"])
    df["CONTRACT"] = secid          # тикер берём из аргумента, не из ответа API
    return df

parts = []
with ThreadPoolExecutor(max_workers=10) as ex:
    for f in as_completed([ex.submit(br_contract_history, s) for s in br_contracts]):
        df = f.result()
        if not df.empty:
            parts.append(df)

allbr = pd.concat(parts, ignore_index=True)
allbr["PRICE"] = allbr["SETTLEPRICE"].fillna(allbr["CLOSE"])
allbr = allbr.dropna(subset=["PRICE", "VOLUME"])

# на каждый день — контракт с максимальным объёмом
idx_max = allbr.groupby("TRADEDATE")["VOLUME"].idxmax()
brent = (allbr.loc[idx_max, ["TRADEDATE", "PRICE", "CONTRACT"]]
              .set_index("TRADEDATE").sort_index())
brent_series = brent["PRICE"].rename("BRENT")
brent_series.to_csv("brent_prices.csv")

print("Brent:", brent_series.shape, "| период:",
      brent_series.index.min().date(), "→", brent_series.index.max().date(),
      "| пропусков:", brent_series.isna().sum())

# Проверка гладкости стыков
brent["roll"] = brent["CONTRACT"] != brent["CONTRACT"].shift()
rolls = brent[brent["roll"]].copy()
rolls["скачок_цены"] = brent["PRICE"].diff()[brent["roll"]]
# Проверка гладкости стыков
print("Макс. скачок цены на стыке, $:", round(rolls["скачок_цены"].abs().max(), 2))
print("Средний |скачок| на стыке, $:", round(rolls["скачок_цены"].abs().mean(), 2))
print("\nПримеры ролловеров (последние 8):")
print(rolls[["CONTRACT", "PRICE", "скачок_цены"]].tail(8).round(2).to_string())

Brent: (1270,) | период: 2021-01-04 → 2025-12-30 | пропусков: 0
Макс. скачок цены на стыке, $: 11.3
Средний |скачок| на стыке, $: 1.93

Примеры ролловеров (последние 8):
           CONTRACT  PRICE  скачок_цены
TRADEDATE                              
2025-05-02     BRM5  61.53        -1.73
2025-06-02     BRN5  64.72         0.82
2025-06-30     BRQ5  66.56        -1.12
2025-08-01     BRU5  69.61        -2.81
2025-09-01     BRV5  68.06        -0.05
2025-10-01     BRX5  65.45        -1.72
2025-11-01     BRZ5  64.77        -0.33
2025-12-01     BRF6  63.38         0.15


# Шаг 6. Фьючерс Brent на 16.06.2025 (ближайший с экспирацией >= +1 мес)

In [15]:
PICK_DAY = pd.Timestamp("2025-06-16")
MIN_EXP  = PICK_DAY + pd.DateOffset(months=1)     # не ближе чем через месяц

# какие BR-фьючерсы торговались в этот день
js = iss_get("/history/engines/futures/markets/forts/securities",
             date=PICK_DAY.strftime("%Y-%m-%d"))
day = pd.DataFrame(js["history"]["data"], columns=js["history"]["columns"])
brday = day[day["ASSETCODE"] == "BR"][["SECID","SHORTNAME","CLOSE","SETTLEPRICE","VOLUME"]].copy()

# узнаем дату экспирации каждого контракта (из описания инструмента)
def last_trade_date(secid):
    d = iss_get(f"/securities/{secid}", **{"iss.only": "description"})["description"]
    desc = pd.DataFrame(d["data"], columns=d["columns"])
    row = desc[desc["name"].str.upper() == "LSTTRADE"]   # дата последнего торгового дня
    return pd.to_datetime(row["value"].iloc[0]) if len(row) else pd.NaT

brday["EXP"] = brday["SECID"].apply(last_trade_date)
brday = brday.sort_values("EXP")
print("BR-фьючерсы на", PICK_DAY.date(), ":")
print(brday.to_string(index=False))

# ближайший с экспирацией не раньше чем через месяц
chosen = brday[brday["EXP"] >= MIN_EXP].iloc[0]
FUT = chosen["SECID"]
print(f"\nВыбран фьючерс: {FUT} ({chosen['SHORTNAME']}), "
      f"экспирация {chosen['EXP'].date()}, цена {chosen['SETTLEPRICE']}")

BR-фьючерсы на 2025-06-16 :
SECID SHORTNAME  CLOSE  SETTLEPRICE    VOLUME        EXP
 BRN5   BR-7.25  72.28        72.14 1211336.0 2025-07-01
 BRQ5   BR-8.25  71.31        71.21  104141.0 2025-08-01
 BRU5   BR-9.25  70.73        70.70    5395.0 2025-09-01
 BRV5  BR-10.25  70.63        70.63     281.0 2025-10-01
 BRX5  BR-11.25  72.93        71.06      68.0 2025-11-03
 BRZ5  BR-12.25  71.98        71.98      88.0 2025-12-01
 BRF6   BR-1.26  72.25        72.25     173.0 2026-01-05
 BRG6   BR-2.26  72.59        72.59     225.0 2026-02-02
 BRH6   BR-3.26  72.84        72.84     106.0 2026-03-02
 BRJ6   BR-4.26  73.37        73.37      68.0 2026-04-01

Выбран фьючерс: BRQ5 (BR-8.25), экспирация 2025-08-01, цена 71.21


In [16]:
opt = iss_paginate("/history/engines/futures/markets/options/securities",
                   "history", date=PICK_DAY.strftime("%Y-%m-%d"))
br_opt = opt[opt["SECID"].astype(str).str.startswith("BR")].copy()
print("Всего опционов в срезе:", len(opt), "| из них BR*:", len(br_opt))

Всего опционов в срезе: 31436 | из них BR*: 836


/tmp/ipykernel_33677/388969069.py:48: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


In [17]:
def option_meta(secid):
    try:
        d = iss_get(f"/securities/{secid}", **{"iss.only": "description"})["description"]
        m = {row[0]: row[2] for row in d["data"]}        # name → value (3-й столбец!)
        strike = m.get("STRIKE")
        return {"SECID": secid,
                "UNDERLYING": m.get("UNDERLYINGASSET"),
                "OPTTYPE":    m.get("OPTIONTYPE"),
                "STRIKE":     float(strike) if strike not in (None, "") else np.nan,
                "LSTTRADE":   m.get("LSTTRADE")}
    except Exception as e:
        return {"SECID": secid, "ERROR": str(e)}          # ошибку не прячем

br_secids = br_opt["SECID"].unique().tolist()
metas = []
with ThreadPoolExecutor(max_workers=12) as ex:
    for f in as_completed([ex.submit(option_meta, s) for s in br_secids]):
        metas.append(f.result())

meta = pd.DataFrame(metas)
errors = meta[meta.get("ERROR").notna()] if "ERROR" in meta.columns else pd.DataFrame()
print("Описаний получено:", len(meta), "| с ошибкой:", len(errors))
if len(errors):
    print("Пример ошибки:", errors.iloc[0].to_dict())

print("\nБазовые активы среди BR-опционов и сколько на каждый:")
print(meta["UNDERLYING"].value_counts())

Описаний получено: 836 | с ошибкой: 0

Базовые активы среди BR-опционов и сколько на каждый:
UNDERLYING
BRN5    176
BRQ5    166
BRZ5     84
BRV5     82
BRU5     82
BRX5     82
BRG6     82
BRF6     82
Name: count, dtype: int64


In [18]:
# Цепочка опционов Put/Call на BRQ5 за 16.06.2025

FUT = "BRQ5"

# только опционы на наш фьючерс
chain = meta[meta["UNDERLYING"] == FUT].copy()

# подтянем расчётную цену (премию) из среза за выбранный день
prices = br_opt.set_index("SECID")["SETTLEPRICE"]
chain["PRICE"] = chain["SECID"].map(prices).astype(float)

# экспирация не ближе месяца от 16.06
chain["LSTTRADE"] = pd.to_datetime(chain["LSTTRADE"])
print("Дата экспирации опционов на BRQ5:", chain["LSTTRADE"].unique(),
      "| порог (≥):", (PICK_DAY + pd.DateOffset(months=1)).date())

# делим на Call и Put
chain = chain.sort_values(["OPTTYPE", "STRIKE"])
calls = chain[chain["OPTTYPE"] == "C"]
puts  = chain[chain["OPTTYPE"] == "P"]
print(f"\nCall: {len(calls)} страйков | Put: {len(puts)} страйков")
print("Диапазон страйков:", chain['STRIKE'].min(), "–", chain['STRIKE'].max(),
      "(цена фьючерса BRQ5 на 16.06 = 71.21)")

# сохраняем
chain_out = chain[["SECID","OPTTYPE","STRIKE","PRICE","LSTTRADE","UNDERLYING"]]
chain_out.to_csv("brent_options_BRQ5_20250616.csv", index=False)
print("\nСохранено опционов:", len(chain_out), "→ brent_options_BRQ5_20250616.csv")

print("\nCall около денег:")
print(calls[(calls.STRIKE >= 65) & (calls.STRIKE <= 78)][["STRIKE","PRICE"]].to_string(index=False))
print("\nPut около денег:")
print(puts[(puts.STRIKE >= 65) & (puts.STRIKE <= 78)][["STRIKE","PRICE"]].to_string(index=False))

Дата экспирации опционов на BRQ5: <DatetimeArray>
['2025-07-28 00:00:00', '2025-07-03 00:00:00']
Length: 2, dtype: datetime64[ns] | порог (≥): 2025-07-16

Call: 83 страйков | Put: 83 страйков
Диапазон страйков: 50.0 – 92.0 (цена фьючерса BRQ5 на 16.06 = 71.21)

Сохранено опционов: 166 → brent_options_BRQ5_20250616.csv

Call около денег:
 STRIKE  PRICE
   65.0   8.25
   65.0   7.17
   66.0   6.44
   66.0   7.43
   67.0   5.78
   67.0   6.65
   68.0   5.18
   68.0   5.90
   69.0   4.62
   69.0   5.19
   70.0   4.11
   70.0   4.53
   71.0   3.66
   71.0   3.92
   72.0   3.25
   72.0   3.36
   73.0   2.90
   73.0   2.86
   74.0   2.59
   74.0   2.42
   75.0   2.32
   75.0   2.04
   76.0   2.09
   76.0   1.72
   77.0   1.90
   77.0   1.46
   78.0   1.73
   78.0   1.26

Put около денег:
 STRIKE  PRICE
   65.0   0.96
   65.0   2.04
   66.0   1.23
   66.0   2.22
   67.0   2.44
   67.0   1.57
   68.0   1.97
   68.0   2.69
   69.0   2.41
   69.0   2.98
   70.0   3.32
   70.0   2.90
   71.0   3.4

In [19]:
MIN_EXP = PICK_DAY + pd.DateOffset(months=1)        # 2025-07-16

# из двух серий берём ту, что удовлетворяет условию (28.07 — основная месячная)
valid_exp = sorted(d for d in chain["LSTTRADE"].unique() if d >= MIN_EXP)
print("Подходящие даты экспирации (≥ порога):", [pd.Timestamp(d).date() for d in valid_exp])

EXP = valid_exp[0]                                  # ближайшая из подходящих
chain_f = chain[chain["LSTTRADE"] == EXP].sort_values(["OPTTYPE","STRIKE"])

calls = chain_f[chain_f["OPTTYPE"] == "C"]
puts  = chain_f[chain_f["OPTTYPE"] == "P"]
print(f"Выбрана экспирация {pd.Timestamp(EXP).date()} | Call: {len(calls)}, Put: {len(puts)}")

chain_out = chain_f[["SECID","OPTTYPE","STRIKE","PRICE","LSTTRADE","UNDERLYING"]]
chain_out.to_csv("brent_options_BRQ5_20250616.csv", index=False)


# по одному опциону на страйк, цены монотонны
print("\nCall около денег:")
print(calls[(calls.STRIKE>=68)&(calls.STRIKE<=74)][["STRIKE","PRICE"]].to_string(index=False))
print("\nPut около денег:")
print(puts[(puts.STRIKE>=68)&(puts.STRIKE<=74)][["STRIKE","PRICE"]].to_string(index=False))

Подходящие даты экспирации (≥ порога): [datetime.date(2025, 7, 28)]
Выбрана экспирация 2025-07-28 | Call: 42, Put: 42

Call около денег:
 STRIKE  PRICE
   68.0   5.18
   69.0   4.62
   70.0   4.11
   71.0   3.66
   72.0   3.25
   73.0   2.90
   74.0   2.59

Put около денег:
 STRIKE  PRICE
   68.0   1.97
   69.0   2.41
   70.0   2.90
   71.0   3.45
   72.0   4.04
   73.0   4.69
   74.0   5.38


In [20]:
indices_df = indices_df[INDICES]          # IMOEX, RTSI
indices_df.index.name = "TRADEDATE"
indices_df.to_csv("indices_prices.csv")

print("Индексы сохранены:", indices_df.shape, "→ indices_prices.csv")
print("Период:", indices_df.index.min().date(), "→", indices_df.index.max().date())
print("\nХвост:")
print(indices_df.tail(3).round(2))

Индексы сохранены: (1254, 2) → indices_prices.csv
Период: 2021-01-04 → 2025-12-30

Хвост:
              IMOEX     RTSI
TRADEDATE                   
2025-12-26  2754.89  1117.04
2025-12-29  2742.03  1115.35
2025-12-30  2766.62  1114.13


# Шаг 7. Объединяем все

In [21]:
# data_raw

def load_csv(path, prefix=""):
    df = pd.read_csv(path, index_col=0, parse_dates=True)
    if prefix:
        df = df.add_prefix(prefix)
    return df

# Загружаем все ряды по пунктам
zcyc_d    = load_csv("zcyc.csv")                       # КБД
ofz_d     = load_csv("ofz_prices.csv")                 # цены ОФЗ
stocks_d  = load_csv("stocks_prices.csv", "STK_")      # акции
indices_d = load_csv("indices_prices.csv", "IDX_")     # индексы
fx_d      = load_csv("fx_prices.csv", "FX_")           # валюты
brent_d   = load_csv("brent_prices.csv")               # Brent

series = {"zcyc": zcyc_d, "ofz": ofz_d, "stocks": stocks_d,
          "indices": indices_d, "fx": fx_d, "brent": brent_d}
for name, df in series.items():
    print(f"{name:<9}{df.shape}  {df.index.min().date()} → {df.index.max().date()}")

# 2Единый торговый календарь
moex_dates = (stocks_d.index.union(ofz_d.index)
              .union(indices_d.index).union(brent_d.index))
master = moex_dates[(moex_dates >= START) & (moex_dates < END)].sort_values()
print(f"\nЕдиный календарь: {len(master)} торговых дней "
      f"({master.min().date()} → {master.max().date()})")

# Приводим все ряды к общей сетке и склеиваем в одну таблицу
raw = pd.concat([df.reindex(master) for df in series.values()], axis=1)
raw.index.name = "DATE"
raw.to_csv("data_raw.csv")

print(f"\ndata_raw собран: {raw.shape} → data_raw.csv")
print("Столбцы:", list(raw.columns))
print("\nДоля пропусков по столбцам, % (топ-8):")
print((raw.isna().mean() * 100).round(2).sort_values(ascending=False).head(8))

zcyc     (1305, 12)  2021-01-01 → 2026-01-01
ofz      (1271, 5)  2021-01-04 → 2025-12-30
stocks   (1271, 10)  2021-01-04 → 2025-12-30
indices  (1254, 2)  2021-01-04 → 2025-12-30
fx       (1237, 2)  2021-01-01 → 2025-12-31
brent    (1270, 1)  2021-01-04 → 2025-12-30

Единый календарь: 1271 торговых дней (2021-01-04 → 2025-12-30)

data_raw собран: (1271, 32) → data_raw.csv
Столбцы: ['ZCYC_0.25y', 'ZCYC_0.5y', 'ZCYC_0.75y', 'ZCYC_1.0y', 'ZCYC_2.0y', 'ZCYC_3.0y', 'ZCYC_5.0y', 'ZCYC_7.0y', 'ZCYC_10.0y', 'ZCYC_15.0y', 'ZCYC_20.0y', 'ZCYC_30.0y', 'OFZ_26219', 'OFZ_26212', 'OFZ_26221', 'OFZ_26218', 'OFZ_26230', 'STK_SBER', 'STK_GAZP', 'STK_LKOH', 'STK_GMKN', 'STK_ROSN', 'STK_NVTK', 'STK_TATN', 'STK_MGNT', 'STK_MTSS', 'STK_CHMF', 'IDX_IMOEX', 'IDX_RTSI', 'FX_USD', 'FX_EUR', 'BRENT']

Доля пропусков по столбцам, % (топ-8):
FX_EUR        23.05
FX_USD        23.05
STK_GMKN       1.73
ZCYC_0.25y     1.57
ZCYC_2.0y      1.57
ZCYC_0.5y      1.57
ZCYC_0.75y     1.57
ZCYC_1.0y      1.57
dtype: float64


In [22]:
# data_clean
clean = raw.copy()

# курс ЦБ действует до следующего объявления
fx_cols = [c for c in clean.columns if c.startswith("FX_")]
clean[fx_cols] = clean[fx_cols].ffill()

# КБД, ОФЗ, акции, индексы, Brent праздники/заморозка подтягиваем последнюю известную цену, лимит 10 дней (покрывает заморозку 2022)
other_cols = [c for c in clean.columns if not c.startswith("FX_")]
clean[other_cols] = clean[other_cols].ffill(limit=10)

# первые строки могли остаться пустыми, если ряд начинается позже — добьём bfill
clean = clean.bfill(limit=3)

clean.index.name = "DATE"
clean.to_csv("data_clean.csv")

before = (raw.isna().mean() * 100).round(2)
after  = (clean.isna().mean() * 100).round(2)
report = pd.DataFrame({"пропуски_до_%": before, "пропуски_после_%": after})
report = report[report["пропуски_до_%"] > 0].sort_values("пропуски_до_%", ascending=False)

print("data_clean собран:", clean.shape, "→ data_clean.csv")
print("\nОтчёт по заполнению пропусков:")
print(report.to_string())
print("\nОстаточные пропуски всего:", int(clean.isna().sum().sum()))

data_clean собран: (1271, 32) → data_clean.csv

Отчёт по заполнению пропусков:
            пропуски_до_%  пропуски_после_%
FX_EUR              23.05              0.16
FX_USD              23.05              0.16
STK_GMKN             1.73              0.24
ZCYC_0.25y           1.57              0.00
ZCYC_2.0y            1.57              0.00
ZCYC_0.5y            1.57              0.00
ZCYC_0.75y           1.57              0.00
ZCYC_1.0y            1.57              0.00
ZCYC_10.0y           1.57              0.00
ZCYC_15.0y           1.57              0.00
ZCYC_20.0y           1.57              0.00
ZCYC_30.0y           1.57              0.00
ZCYC_5.0y            1.57              0.00
ZCYC_3.0y            1.57              0.00
ZCYC_7.0y            1.57              0.00
STK_SBER             1.42              0.24
IDX_RTSI             1.42              0.24
STK_TATN             1.42              0.24
STK_MGNT             1.42              0.24
STK_MTSS             1.42              0.

In [23]:
clean = clean.ffill().bfill()
clean.to_csv("data_clean.csv")
print("Остаточные пропуски после финальной зачистки:", int(clean.isna().sum().sum()))

Остаточные пропуски после финальной зачистки: 0
